# 01 – Análise Exploratória e Montagem de Base

Carrega o `.xlsx`, identifica planilhas de vendas e pagamentos, tipa colunas, constrói cronograma de parcelas e aloca pagamentos.

In [1]:
import sys, os
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))

In [2]:
from pathlib import Path
import pandas as pd
from src.config import ensure_dirs, RAW_DIR, PROC_DIR, DEFAULT_FILES, get_events
from src.io_utils import copy_raw_files
from src.loaders import load_base
from src.transform import build_schedule, assign_payments
ensure_dirs()
paths = copy_raw_files()
xlsx = paths.get('xlsx', DEFAULT_FILES['xlsx'])
sales, payments = load_base(xlsx)
sales.to_csv(PROC_DIR / 'sales_clean.csv', index=False)
payments.to_csv(PROC_DIR / 'payments_clean.csv', index=False)
schedule = build_schedule(sales)
allocated = assign_payments(schedule, payments)
schedule.to_csv(PROC_DIR / 'schedule.csv', index=False)
allocated.to_csv(PROC_DIR / 'allocated.csv', index=False)
events = get_events()
events

C:\Users\flavi\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


{'high_ticket_start': datetime.date(2021, 1, 1),
 'recurring_expansion': datetime.date(2021, 7, 1),
 'premium_offer': datetime.date(2024, 8, 1),
 'whatsapp_start': datetime.date(2020, 2, 1)}

In [3]:
sales.info()
sales.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 169591 entries, 0 to 169590
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   contract_id         169591 non-null  str           
 1   customer_id         169591 non-null  str           
 2   purchase_date       169591 non-null  datetime64[us]
 3   product_name        169591 non-null  str           
 4   installments_total  169591 non-null  int32         
 5   installment_value   169591 non-null  float64       
 6   recurring           169591 non-null  bool          
 7   interest_rate       0 non-null       object        
 8   product_price       169591 non-null  float64       
dtypes: bool(1), datetime64[us](1), float64(2), int32(1), object(1), str(3)
memory usage: 25.1+ MB


,contract_id,customer_id,purchase_date,product_name,installments_total,installment_value,recurring,interest_rate,product_price
count,169591,169591,169591,169591,169591.000000,1.695910e+05,169591,0,1.695910e+05
unique,169591,130821,NaN,8,NaN,NaN,2,0,NaN
top,TS11111544818515,gabriela.rezende34@gmail.com,NaN,Fim de semana em resort all-inclusive em Búzios,NaN,NaN,False,NaN,NaN
freq,1,9,NaN,75266,NaN,NaN,119457,NaN,NaN
mean,NaN,NaN,2023-09-03 05:03:48.005023,NaN,8.631773,8.151950e+02,NaN,NaN,6.442998e+03
min,NaN,NaN,2020-01-04 00:00:00,NaN,1.000000,2.000000e+00,NaN,NaN,2.000000e+00
25%,NaN,NaN,2022-05-20 00:00:00,NaN,3.000000,1.495000e+02,NaN,NaN,9.970000e+02
50%,NaN,NaN,2023-11-08 00:00:00,NaN,12.000000,6.818000e+02,NaN,NaN,2.388000e+03
75%,NaN,NaN,2025-01-11 00:00:00,NaN,12.000000,1.085640e+03,NaN,NaN,8.496000e+03
max,NaN,NaN,2025-12-31 00:00:00,NaN,24.000000,2.231447e+06,NaN,NaN,9.666288e+06


In [4]:
payments.info()
payments.describe(include='all')

<class 'pandas.DataFrame'>
Index: 469461 entries, 0 to 469467
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   payment_date    469461 non-null  datetime64[us]
 1   payment_amount  469461 non-null  float64       
 2   ref_id          469461 non-null  str           
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 21.5 MB


,payment_date,payment_amount,ref_id
count,469461,4.694610e+05,469461
unique,NaN,NaN,169591
top,NaN,NaN,TS33855305321396
freq,NaN,NaN,12
mean,2024-04-04 18:05:19.172838,3.703692e+02,NaN
min,2020-01-04 00:00:00,2.000000e+00,NaN
25%,2023-06-21 00:00:00,9.714000e+01,NaN
50%,2024-07-30 00:00:00,9.714000e+01,NaN
75%,2025-05-09 00:00:00,3.490000e+02,NaN
max,2025-12-31 00:00:00,2.231447e+06,NaN


In [5]:
nulls_sales = sales.isnull().mean().sort_values(ascending=False)
nulls_pay = payments.isnull().mean().sort_values(ascending=False)
nulls_sales, nulls_pay

(interest_rate         1.0
 contract_id           0.0
 customer_id           0.0
 purchase_date         0.0
 product_name          0.0
 installments_total    0.0
 installment_value     0.0
 recurring             0.0
 product_price         0.0
 dtype: float64,
 payment_date      0.0
 payment_amount    0.0
 ref_id            0.0
 dtype: float64)